# Notebook 02 — Simulação de Auto-Scaling e Canary Deployment

## Objetivos
1. Simular auto-scaling reativo baseado em métricas de CPU e latência.
2. Aplicar teoria de filas (Erlang-C) para dimensionamento de réplicas.
3. Orquestrar canary deployment progressivo com validação estatística.
4. Simular pipeline multimodal com `MultimodalOrchestrator`.

## Conceito-chave (Documento 04)
> "O auto-scaling para inferência de ML apresenta desafios específicos que merecem atenção.
> O tempo de cold start, que inclui carregamento do modelo em memória (potencialmente GBs
> para modelos grandes) e warm-up de caches, pode ser substancial. Estratégias de mitigação
> incluem manter um número mínimo de réplicas sempre ativas."
> — Seção "Azure ML Managed Endpoints: Auto-scaling em Produção"

### Teoria de Filas — Erlang-C
$$W_q = \frac{C(c, \rho)}{c\mu - \lambda}$$
onde $\rho = \lambda / (c \cdot \mu)$ é a utilização do sistema.

In [ ]:
# Imports
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing import DataPreprocessor
from src.model import SLOMonitor, CanaryAnalyzer, MultimodalOrchestrator
from src.training import (
    AutoScaler, CanaryDeployment,
    run_autoscaling_simulation, cross_validate_scaling_params,
    hyperparameter_tuning,
)
from src.evaluation import plot_scaling_timeline, plot_canary_comparison

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Carregar Dados

In [ ]:
dp = DataPreprocessor(seed=42)
data_path = os.path.join("..", "data", "raw", "dataset.csv")
df = dp.load_data(data_path)
df = dp.clean_data(df)
df = dp.prepare_features(df)
print(f"Dataset: {df.shape}")

## 2. Teoria de Filas — Erlang-C

Antes de simular o auto-scaler, visualizamos como a latência teórica
varia com o número de réplicas, usando a fórmula Erlang-C apresentada
na seção *Azure ML Managed Endpoints* do material.

Parâmetros de exemplo:
- λ (taxa de chegada) = 100 req/s
- μ (taxa de serviço por réplica) = 20 req/s

In [ ]:
lam = 100  # req/s
mu = 20    # req/s por réplica

replicas_range = range(6, 16)
wq_values = []

for c in replicas_range:
    wq = SLOMonitor.mean_queue_wait(lam, mu, c)
    wq_values.append(wq * 1000)  # converter para ms

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(list(replicas_range), wq_values, marker="o", linewidth=2, color="steelblue")
ax.axhline(200, color="red", linestyle="--", label="SLO = 200 ms")
ax.set_xlabel("Número de Réplicas (c)")
ax.set_ylabel("Wq — Tempo Médio de Espera (ms)")
ax.set_title("Erlang-C: Latência vs Réplicas (λ=100, μ=20)")
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.show()

print("\nLatência teórica por número de réplicas:")
for c, wq in zip(replicas_range, wq_values):
    status = "✓ SLO" if wq <= 200 else "✗ Violação"
    print(f"  c={c:2d} → Wq = {wq:10.2f} ms  [{status}]")

## 3. Simulação de Auto-Scaling

Simulamos um auto-scaler reativo que monitora CPU e latência P99
para decidir scale-up/scale-down, conforme configuração descrita
no Snippet 1 (Azure ML Managed Endpoint) do material:

```python
endpoint = ManagedOnlineEndpoint(
    name="multimodal-endpoint",
    tags={"slo_p99_ms": "200", "error_budget": "0.1"}
)
```

In [ ]:
df_sim, events = run_autoscaling_simulation(
    df,
    min_replicas=2,
    max_replicas=10,
    cpu_upper=0.70,
    latency_target_ms=200.0,
)

print(f"Eventos de scaling: {len(events)}")
print(f"  Scale-up:   {sum(1 for e in events if e.direction == 'scale_up')}")
print(f"  Scale-down: {sum(1 for e in events if e.direction == 'scale_down')}")
print(f"Réplicas — min: {df_sim['simulated_replicas'].min()}, "
      f"max: {df_sim['simulated_replicas'].max()}, "
      f"média: {df_sim['simulated_replicas'].mean():.1f}")

In [ ]:
# Visualização da timeline de auto-scaling
fig = plot_scaling_timeline(
    df_sim, events,
    save_path=os.path.join("..", "outputs", "figures", "scaling_timeline.png")
)
plt.show()

## 4. Tuning de Parâmetros de Auto-Scaling

Exploramos diferentes configurações de min/max réplicas e threshold
de CPU, avaliando o trade-off custo × latência.

In [ ]:
results = cross_validate_scaling_params(df)
results_df = pd.DataFrame(results)

# Top 5 configs por custo (mantendo SLO)
valid = results_df[results_df["latency_p99"] <= 200].sort_values("cost_score")
print("Top 5 configurações (custo mínimo com SLO atendido):")
valid.head()

In [ ]:
best = hyperparameter_tuning(df)
print(f"Melhor configuração: {best}")

## 5. Canary Deployment com Validação Estatística

Implementa a Tabela 2 do material — *Framework de Decisão para Canary Deployment*:

| Métrica | Teste | Critério de Rollback |
|---------|-------|---------------------|
| Latência P99 | t-test | p < 0.05 e degradação > 10 % |
| Taxa de Erro | χ² | p < 0.05 e aumento > 50 % |

O canary deployment progressivo avança por etapas (10 % → 25 % → 50 % → 100 %),
conforme descrito no Snippet 2 do material:

```python
endpoint.traffic = {"model-v1-stable": 90, "model-v2-canary": 10}
```

In [ ]:
canary_deploy = CanaryDeployment(
    stages=[0.10, 0.25, 0.50, 1.0],
    analyzer=CanaryAnalyzer(alpha=0.05),
)

stage_results = canary_deploy.run(df)

print("=== Canary Deployment — Resultados por Etapa ===")
for sr in stage_results:
    print(f"\nEtapa {sr.stage} (tráfego canary: {sr.traffic_pct*100:.0f}%)")
    print(f"  Requisições: {sr.n_requests}")
    print(f"  Latência  — stat={sr.latency_result.statistic:.3f}, "
          f"p={sr.latency_result.p_value:.4f}, decisão={sr.latency_result.decision}")
    print(f"  Error Rate — stat={sr.error_result.statistic:.3f}, "
          f"p={sr.error_result.p_value:.4f}, decisão={sr.error_result.decision}")
    print(f"  >>> Decisão geral: {sr.overall_decision.upper()}")

In [ ]:
# Comparação visual stable vs canary
fig = plot_canary_comparison(
    df,
    save_path=os.path.join("..", "outputs", "figures", "canary_comparison.png")
)
plt.show()

## 6. Simulação de Pipeline Multimodal

Utilizamos o `MultimodalOrchestrator` para simular a arquitetura de
referência (Figura 1): API Gateway → Workers → Aggregator (late fusion).

Conforme discutido na seção *Orquestração de Pipelines Multimodais*:
> "Late fusion processa cada modalidade independentemente e combina
> predições finais através de voting, averaging, ou modelos de meta-learning."

In [ ]:
orchestrator = MultimodalOrchestrator(fusion_strategy="late", seed=42)

# Simular 5 requisições de exemplo
for i in range(5):
    result = orchestrator.predict(modalities=["text", "image", "audio"])
    print(f"Req {i+1}: latência={result['total_latency_ms']:.1f}ms, "
          f"confidence={result['fused_confidence']:.3f}, "
          f"erro={result['has_error']}")

In [ ]:
# Métricas agregadas do orchestrator
scores = orchestrator.score(n_requests=500)
print("\n=== Métricas do Orchestrator (500 req) ===")
for k, v in scores.items():
    print(f"  {k:20s}: {v:.4f}")

## Resumo

Neste notebook:
- Exploramos a teoria de filas (Erlang-C) e seu impacto no dimensionamento.
- Simulamos auto-scaling reativo e avaliamos diferentes configurações.
- Executamos canary deployment progressivo com testes estatísticos.
- Simulamos o pipeline multimodal completo (text + image + audio).

**Próximo:** [03_avaliacao.ipynb](03_avaliacao.ipynb) — Avaliação de SLOs, error budgets e dashboard de observabilidade.